# Advanced Predictive Models in R: Lasso, Trees, Random Forests, and Boosting

---
This notebook shows how to use more advanced machine learning models in R for predicting customer behaviour in banking. We use the same bank marketing data as in Units 01 and 02: the outcome `y` is 1 if the customer subscribed to a term deposit after the call. We cover:

- **Lasso regression**: A linear model that automatically selects important features
- **Decision Trees**: Easy-to-interpret models that make decisions like a flowchart
- **Random Forests**: Combines many decision trees for better predictions
- **Boosting**: Builds models sequentially, learning from previous mistakes

## Learning Objectives
By the end of this notebook, you will be able to:
1. Choose a feature set with the decision-time rule and explain why `duration` is dropped.
2. Fit a cross-validated Lasso, a shallow tree, a random forest and a boosted model on one train/test split.
3. Show, with numbers, why a single deep tree overfits and why averaging many trees helps.
4. Compare models with accuracy, AUC, recall and precision against the majority-class baseline, and read the results as a call-centre manager would.
5. Explain the difference between "important for prediction" and "causes subscription", and state what a causal estimate would need to be believed.

## Required Packages
The first code cell installs what is missing as pre-built binaries from the Posit Package Manager mirror, which takes under a minute on Google Colab (if the mirror has no binary for a package, R falls back to building it from source, which can take 15 to 30 minutes). Section 6 (causal analysis) is optional; it installs one more package, `grf`, from the same mirror and takes about five minutes to run.


In [ ]:
# Load the packages, installing the ones that are missing.
# Pre-built binaries from the Posit Package Manager keep the installs fast on Colab (Ubuntu 22.04, "jammy").
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
required_packages <- c("data.table", "ggplot2", "gamlr", "rpart", "randomForest", "xgboost", "pROC")

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    cat("Installing package:", pkg, "\n")
    install.packages(pkg)
  }
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}

cat("All packages loaded.\n")


In [ ]:
# Load and explore the banking dataset (fread reads directly from the URL)
data <- fread("https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv")

cat("Dataset dimensions:", nrow(data), "rows and", ncol(data), "columns\n\n")
str(data)

cat("\nMissing values per column:\n")
print(sapply(data, function(x) sum(is.na(x))))

# Look at the target variable distribution
cat("\nTarget variable (y) distribution:\n")
print(table(data$y))
cat("\nShare of subscribers:", round(mean(data$y), 3), "\n")


## Data Preparation

Before building models we choose the feature set and make one train/test split that every model in this notebook will use.

**Which columns are known at decision time?** The question a model in this unit answers is: "before we call this customer, how likely is it that they subscribe?" A feature is allowed only if the bank knows it at that moment.

- `duration` is the length of the call. It is known only after the call ends, and a call of length zero cannot end in a subscription. Unit 02 dropped it for the same reason (leakage). We drop it here too. Section 3 has an optional demonstration of what happens if you keep it.
- `campaign` (contacts so far in this campaign), `pdays`, `previous` and `poutcome` (history of earlier campaigns) are in the customer file before the call, so they stay.
- The five macro indicators (`emp_var_rate` to `nr_employed`) are published figures and stay.

The split is a simple random 80/20 split; the Python notebook stratifies on `y` instead, so the two notebooks' test sets differ slightly. We convert the text columns to factors and recode `y` once, from 0/1 to a factor with levels `no` and `yes`, because `rpart` and `randomForest` read a factor outcome as a classification task. `gamlr` and `xgboost` need a numeric design matrix and a numeric 0/1 outcome instead, so we build those once here as well. The same helper function scores every model on the same test set, so the comparison in Section 5 is fair.


In [ ]:
# Factors for the text columns; y recoded once to a factor with levels "no" / "yes"
categorical_vars <- c("marital", "education", "housing", "loan", "contact", "poutcome")
data[, (categorical_vars) := lapply(.SD, as.factor), .SDcols = categorical_vars]
data[, y := factor(y, levels = c(0, 1), labels = c("no", "yes"))]

# Feature set: everything the bank knows before the call
data_full <- data.table::copy(data)     # kept only for the optional demonstrations later on
data[, duration := NULL]

# One train/test split (80/20)
set.seed(123)
train_idx <- sample(seq_len(nrow(data)), size = 0.8 * nrow(data))
train_data <- data[train_idx, ]
test_data  <- data[-train_idx, ]
cat("Training set:", nrow(train_data), "observations\n")
cat("Test set:", nrow(test_data), "observations\n")

# Numeric design matrices (dummy-encoded) for gamlr and xgboost; the tree models take the data.table directly
X_train <- model.matrix(y ~ ., train_data)[, -1]    # drop the intercept column
X_test  <- model.matrix(y ~ ., test_data)[, -1]
y_train <- as.numeric(train_data$y == "yes")         # 0/1 outcome for gamlr and xgboost
y_test  <- as.numeric(test_data$y == "yes")
cat("\nNumber of features after dummy encoding:", ncol(X_train), "\n")

# The baseline every model must beat: always predict "no"
majority_rate <- mean(y_test == 0)
cat("\nShare of subscribers in the test set:", round(mean(y_test), 3), "\n")
cat("Majority-class baseline accuracy (predict 'no' for everyone):", round(majority_rate, 3), "\n")

# One evaluation helper used for every model, so all models are scored the same way
results <- list()   # model name -> one-row data.frame of metrics, filled in as we go

auc_of <- function(truth, prob) {
  as.numeric(pROC::auc(pROC::roc(truth, prob, levels = c(0, 1), direction = "<", quiet = TRUE)))
}

evaluate <- function(name, prob, threshold = 0.5) {
  prob <- as.numeric(prob)
  pred <- as.numeric(prob > threshold)
  tp <- sum(pred == 1 & y_test == 1)
  fp <- sum(pred == 1 & y_test == 0)
  fn <- sum(pred == 0 & y_test == 1)
  m <- data.frame(
    Model = name,
    Accuracy = mean(pred == y_test),
    AUC = auc_of(y_test, prob),
    Recall_yes = ifelse(tp + fn > 0, tp / (tp + fn), 0),
    Precision_yes = ifelse(tp + fp > 0, tp / (tp + fp), NA_real_)
  )
  results[[name]] <<- m
  cat(sprintf("\n%s on the test set (threshold %.1f):\n", name, threshold))
  cat(sprintf("  Accuracy:  %.3f   (majority baseline: %.3f)\n", m$Accuracy, majority_rate))
  cat(sprintf("  AUC:       %.3f   (random guessing: 0.500)\n", m$AUC))
  cat(sprintf("  Recall on subscribers:    %.3f   (share of actual subscribers the model flags)\n", m$Recall_yes))
  cat(sprintf("  Precision on subscribers: %.3f   (share of flagged customers who subscribe)\n", m$Precision_yes))
  cat("  Confusion matrix:\n")
  print(table(Predicted = factor(pred, levels = c(0, 1)), Actual = factor(y_test, levels = c(0, 1))))
  invisible(m)
}


## 1. Lasso Regression

**What is Lasso?** Lasso (Least Absolute Shrinkage and Selection Operator) is a regression (here a logistic regression) with a penalty on the absolute size of the coefficients. The penalty pushes small coefficients exactly to zero, so the model selects features on its own. That reduces overfitting and makes the model easier to read.

**When to use Lasso:**
- When you have many features and want automatic feature selection
- When you need an interpretable model
- When you suspect many features are irrelevant

**Implementation details:**
- `gamlr` standardises the features internally, so every coefficient is penalised on the same scale, and reports the coefficients on the original scale.
- `cv.gamlr` fits the whole path of penalties (lambda; a large lambda means a strong penalty) and picks the lambda with the lowest cross-validated error (`lambda.min`). It also reports `lambda.1se`, the strongest penalty whose error is within one standard error of the best, which prefers the simpler model when the curve is flat.
- The response must be numeric 0/1 for `family = "binomial"`, which is why the data preparation built `y_train` that way.

### Logistic Lasso: Predicting customer subscription (y)


In [ ]:
# Fit the logistic lasso path with 5-fold cross-validation
set.seed(123)
cv_lasso <- cv.gamlr(X_train, y_train, family = "binomial", nfold = 5)

plot(cv_lasso, main = "Lasso Cross-Validation")
cat("Lambda with the lowest CV error (lambda.min):", signif(cv_lasso$lambda.min, 4), "\n")
cat("Strongest lambda within one SE of it (lambda.1se):", signif(cv_lasso$lambda.1se, 4), "\n")

# Number of non-zero coefficients along the path (the intercept is not counted)
path_nonzero <- colSums(as.matrix(cv_lasso$gamlr$beta) != 0)
plot(log(cv_lasso$gamlr$lambda), path_nonzero, type = "b", pch = 16,
     xlab = "log(lambda)  (strong penalty on the right)", ylab = "Non-zero coefficients",
     main = "Lasso path: how many features survive as the penalty weakens")
abline(v = log(cv_lasso$lambda.min), lty = 2, col = "grey40")
abline(v = log(cv_lasso$lambda.1se), lty = 3, col = "grey40")
legend("topright", c("lambda.min", "lambda.1se"), lty = c(2, 3), col = "grey40")

# Coefficients at lambda.min (the model we evaluate below)
lasso_coef <- coef(cv_lasso, select = "min")
cat("\nNumber of selected features at lambda.min:", sum(lasso_coef[-1, ] != 0), "out of", ncol(X_train), "\n")
cat("Number of selected features at lambda.1se:", sum(coef(cv_lasso, select = "1se")[-1, ] != 0), "\n")

selected_features <- lasso_coef[lasso_coef[, 1] != 0, , drop = FALSE]
cat("\nNon-zero coefficients at lambda.min (original scale), largest first:\n")
print(selected_features[order(-abs(selected_features[, 1])), , drop = FALSE])

# Predictions on the test set, from the same lambda.min model as the coefficients above
lasso_pred_prob <- as.numeric(predict(cv_lasso, X_test, type = "response", select = "min"))
evaluate("Lasso", lasso_pred_prob)


## 2. Decision Trees

**What are Decision Trees?** Decision trees make predictions by asking a series of yes/no questions about the features. They are like a flowchart that leads to a prediction.

**When to use Decision Trees:**
- When you need a highly interpretable model
- When relationships between features are non-linear
- When you want to understand the decision-making process

**Pros:** Easy to read, no scaling needed, captures non-linear patterns and interactions.
**Cons:** A tree grown without limits memorises the training data (overfits), and small changes in the data can produce a very different tree.

We first fit a small tree with at most three levels and at least 100 customers per leaf, so that we can read it. `rpart`'s default stopping rule is a relative cost-complexity parameter `cp`, which has no exact counterpart in scikit-learn; we switch it off (`cp = 0`) and use the same depth and leaf-size limits in both languages. One difference remains: `rpart` keeps a split only if it lowers the number of misclassified customers, so a split that separates "unlikely to subscribe" from "very unlikely to subscribe" is dropped even at `cp = 0`. scikit-learn keeps such splits because they reduce impurity. The R tree can therefore end up with fewer leaves than the Python tree, and a lower AUC, from the same limits.


In [ ]:
# A shallow, readable tree
tree_mod <- rpart(y ~ ., data = train_data, method = "class",
                  control = rpart.control(maxdepth = 3, minbucket = 100, cp = 0, xval = 0))

plot(tree_mod, uniform = TRUE, margin = 0.1)
text(tree_mod, use.n = TRUE, all = TRUE, cex = 0.8)
title("Decision Tree for Customer Subscription Prediction (depth 3)")
print(tree_mod)

tree_pred_prob <- predict(tree_mod, test_data, type = "prob")[, "yes"]
evaluate("Decision Tree", tree_pred_prob)

# Feature importance: impurity reduction attributed to each feature.
# It tells you which features the tree used to sort customers, not how much they change the probability.
cat("\nFeature Importance (Decision Tree):\n")
print(round(sort(tree_mod$variable.importance, decreasing = TRUE), 1))


### Why we limit the tree: overfitting

What happens if we let the tree grow until every leaf is pure? Compare the AUC on the training data with the AUC on the test data, for the shallow tree and for an unlimited tree.


In [ ]:
# An unlimited tree: no complexity penalty, a leaf may hold a single customer
deep_tree <- rpart(y ~ ., data = train_data, method = "class",
                   control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, maxdepth = 30, xval = 0))

tree_auc <- function(model, newdata, truth) auc_of(truth, predict(model, newdata, type = "prob")[, "yes"])

overfit_table <- data.frame(
  Model = c("Shallow tree (depth 3, leaf >= 100)", "Unlimited tree"),
  Leaves = c(sum(tree_mod$frame$var == "<leaf>"), sum(deep_tree$frame$var == "<leaf>")),
  Train_AUC = c(tree_auc(tree_mod, train_data, y_train), tree_auc(deep_tree, train_data, y_train)),
  Test_AUC = c(tree_auc(tree_mod, test_data, y_test), tree_auc(deep_tree, test_data, y_test))
)
overfit_table$Train_AUC <- round(overfit_table$Train_AUC, 3)
overfit_table$Test_AUC <- round(overfit_table$Test_AUC, 3)
print(overfit_table, row.names = FALSE)


**What the table shows.** The unlimited tree has thousands of leaves and a training AUC close to 1, but a test AUC well below the shallow tree's. It has memorised the training customers, noise included, and that memory does not transfer to new customers. The gap between training and test performance is what overfitting means. Limiting depth and leaf size is one cure. Section 3 shows another: keep the deep trees, but average many of them.


## 3. Random Forests

**What are Random Forests?** A random forest grows many deep trees, each on a bootstrap sample of the training rows, and at each split lets a tree choose among a random subset of the features only. The forest's prediction is the average of the trees' predictions.

**Why does averaging help?** Section 2 showed that a single deep tree has low bias but high variance: it fits the training data almost perfectly and generalises poorly. Averaging many such trees keeps the low bias and cuts the variance, in the same way that the average of many noisy measurements is less noisy than one measurement. Bootstrap samples and random feature subsets make the trees different from each other, and averaging only reduces variance when the trees do not all make the same mistakes.

**When to use Random Forests:**
- When you want better accuracy than a single decision tree with little tuning
- When you have enough data (thousands of rows)
- When you want a feature importance ranking

**Pros:** Usually more accurate than a single tree, few parameters to tune, and the out-of-bag (OOB) error gives an honest performance estimate without a test set.
**Cons:** Not readable as a flowchart, slower to train and predict than a single tree, and the `randomForest` package refuses rows with missing values (this data set has none).


In [ ]:
# Fit a random forest for classification
set.seed(123)
mtry <- floor(sqrt(ncol(train_data) - 1))   # features tried at each split; the usual default for classification
rf_mod <- randomForest(y ~ ., data = train_data, ntree = 500, mtry = mtry, importance = TRUE)

print(rf_mod)   # includes the out-of-bag error rate and confusion matrix
oob_accuracy <- 1 - rf_mod$err.rate[rf_mod$ntree, "OOB"]
cat(sprintf("\nOut-of-bag accuracy: %.3f   (majority baseline in training data: %.3f)\n", oob_accuracy, mean(y_train == 0)))
cat("(each tree is scored on the training rows it did not see, so this needs no test set)\n")

rf_pred_prob <- predict(rf_mod, test_data, type = "prob")[, "yes"]
evaluate("Random Forest", rf_pred_prob)

# Feature importance (mean decrease in Gini impurity, summed over all trees)
importance_rf <- importance(rf_mod)[, "MeanDecreaseGini"]
cat("\nTop 10 Most Important Features (Random Forest):\n")
print(round(sort(importance_rf, decreasing = TRUE)[1:10], 1))

varImpPlot(rf_mod, main = "Random Forest Feature Importance")


**Reading the importance ranking.** Feature importance measures how much a feature helped the forest sort customers into subscribers and non-subscribers. It is not an effect size: a high-ranked feature is not "the reason" a customer subscribes, and the ranking says nothing about what would happen if the bank changed that feature. "Important for prediction" and "causes subscription" are different claims. Section 6 returns to this.


### Why many trees: variance reduction

How does the test AUC change as trees are added? We refit the forest with 1, 5, 25, 100 and 500 trees. A forest with one tree is just one deep tree from Section 2, grown on a bootstrap sample.


In [ ]:
n_trees_grid <- c(1, 5, 25, 100, 500)
auc_by_trees <- numeric(length(n_trees_grid))
for (i in seq_along(n_trees_grid)) {
  set.seed(123)
  rf_n <- randomForest(y ~ ., data = train_data, ntree = n_trees_grid[i], mtry = mtry)
  auc_by_trees[i] <- auc_of(y_test, predict(rf_n, test_data, type = "prob")[, "yes"])
  cat(sprintf("%4d trees: test AUC = %.3f\n", n_trees_grid[i], auc_by_trees[i]))
}

plot(n_trees_grid, auc_by_trees, type = "b", pch = 16, log = "x",
     xlab = "Number of trees (log scale)", ylab = "Test AUC",
     main = "Random forest: test AUC as trees are added")
grid()


**What the curve shows.** The test AUC rises steeply over the first few trees and then flattens; almost all of the gain comes early, after which more trees only smooth the estimate. Three things make this work. *Bagging:* each tree sees a different bootstrap sample, so the trees' errors are partly independent and averaging cancels them. *Feature subsampling:* at each split a tree may only choose among `mtry` randomly picked features, which stops every tree from opening with the same strongest feature, makes the trees less alike, and so lets averaging remove more variance. *Averaging itself:* each single tree is as overfitted as the unlimited tree of Section 2, yet the average of 500 of them is not, because the noise each tree fitted is different.


### Optional: what if we had kept `duration`?

The data preparation dropped `duration` because the bank does not know the call length before the call. Here is what the forest would report if we had kept it.


In [ ]:
# Refit the forest with duration included (same split, same rows)
train_full <- data_full[train_idx, ]
test_full  <- data_full[-train_idx, ]

set.seed(123)
rf_dur <- randomForest(y ~ ., data = train_full, ntree = 200, mtry = mtry)
auc_with <- auc_of(y_test, predict(rf_dur, test_full, type = "prob")[, "yes"])
auc_without <- results[["Random Forest"]]$AUC

cat(sprintf("Test AUC without duration: %.3f\n", auc_without))
cat(sprintf("Test AUC with duration:    %.3f\n", auc_with))
cat(sprintf("Gap:                       %+.3f\n", auc_with - auc_without))

cat("\nTop 3 features with duration included:\n")
print(round(sort(importance(rf_dur)[, "MeanDecreaseGini"], decreasing = TRUE)[1:3], 1))


**Why is the "better" model useless at decision time?** With `duration` included the AUC jumps and `duration` becomes the top feature by a wide margin. But the input the model now relies on does not exist when the decision is made: the bank picks whom to call before the call, and the length of that call is a result of the call, not a property of the customer. A model scored with information from the future looks excellent in a backtest and cannot be run in production. Whenever a feature makes a model look too good, ask when that feature becomes known.


## 4. Boosting (using XGBoost)

**What is Gradient Boosting?** Boosting also combines many trees, but not by averaging independent trees. It builds small trees one after another, and each new tree is fitted to the errors the current ensemble still makes. The learning rate `eta` scales down each tree's contribution, so the model improves in many small steps. XGBoost is a fast and widely used implementation.

**Early stopping.** Because every round adds a tree that fits the remaining errors, a boosted model keeps improving on the training data and at some point starts to overfit. `xgb.cv` therefore tracks the AUC on held-out folds after every round and stops when it has not improved for 10 rounds. We then refit on all training data with that number of rounds (`xgb.train`, whose interface is stable across `xgboost` versions).

**When to use Boosting:**
- When you want the best predictive performance on tabular data
- When you can afford to tune the learning rate, the tree depth and the number of rounds
- When accuracy matters more than interpretability

**Pros:** Often the most accurate model on tabular data; fast; handles missing values natively; reports a gain-based feature importance.
**Cons:** More parameters to tune, sensitive to the learning rate, not readable, and it needs a numeric design matrix rather than a data frame with factors.


In [ ]:
# xgboost needs numeric matrices and a numeric 0/1 label (built in the data preparation)
dtrain <- xgb.DMatrix(data = X_train, label = y_train)
dtest  <- xgb.DMatrix(data = X_test, label = y_test)

params <- list(
  objective = "binary:logistic",
  eval_metric = "auc",
  max_depth = 4,
  eta = 0.1,
  subsample = 0.8,
  colsample_bytree = 0.8
)

# Cross-validation with early stopping to find the number of rounds
set.seed(123)
cv_result <- xgb.cv(params = params, data = dtrain, nrounds = 500, nfold = 5,
                    early_stopping_rounds = 10, verbose = 0)

# The best iteration is stored in different places depending on the xgboost version; read it defensively
best_nrounds <- cv_result$best_iteration
if (is.null(best_nrounds)) best_nrounds <- cv_result$early_stop$best_iteration
if (is.null(best_nrounds)) best_nrounds <- which.max(cv_result$evaluation_log$test_auc_mean)
cat("Rounds allowed: 500; best number of rounds from cross-validation:", best_nrounds, "\n")

# Training vs held-out AUC by round: what early stopping is looking at
cv_log <- as.data.frame(cv_result$evaluation_log)
plot(cv_log$iter, cv_log$train_auc_mean, type = "l", lwd = 2, col = "steelblue",
     ylim = range(c(cv_log$train_auc_mean, cv_log$test_auc_mean)),
     xlab = "Boosting round", ylab = "AUC (5-fold CV)",
     main = "XGBoost: training vs held-out AUC by round")
lines(cv_log$iter, cv_log$test_auc_mean, lwd = 2, col = "darkorange")
abline(v = best_nrounds, lty = 2, col = "grey40")
legend("bottomright", c("Training folds", "Held-out folds", "Best round"),
       col = c("steelblue", "darkorange", "grey40"), lty = c(1, 1, 2), lwd = 2)

# Refit on all training data with the chosen number of rounds
xgb_mod <- xgb.train(params = params, data = dtrain, nrounds = best_nrounds, verbose = 0)

xgb_pred_prob <- predict(xgb_mod, dtest)
evaluate("XGBoost", xgb_pred_prob)

# Feature importance (gain: how much each feature improved the fit when it was used to split)
importance_xgb <- xgb.importance(model = xgb_mod)
cat("\nTop 10 Most Important Features (XGBoost, by gain):\n")
print(head(importance_xgb, 10))
xgb.plot.importance(head(importance_xgb, 15), main = "XGBoost Feature Importance (Top 15)")

# ROC curve
roc_obj <- roc(y_test, xgb_pred_prob, levels = c(0, 1), direction = "<", quiet = TRUE)
plot(roc_obj, main = sprintf("ROC Curve (XGBoost, AUC = %.3f)", as.numeric(auc(roc_obj))))


## 5. Model Comparison

All four models are scored on the same test set at the same threshold of 0.5. The first row is the baseline that predicts "no" for everyone.

- **Accuracy** is the share of correct predictions. With 11 percent subscribers, "always no" already gets 89 percent, so accuracy alone cannot tell the models apart.
- **AUC** is the probability that a randomly chosen subscriber gets a higher predicted probability than a randomly chosen non-subscriber. It does not depend on the threshold and is the main number to compare here.
- **Recall on subscribers** is the share of actual subscribers the model flags. For a call centre: of all customers who would say yes, how many does the model put on the call list?
- **Precision on subscribers** is the share of flagged customers who subscribe. For a call centre: of the customers on the list, how many calls end in a sale?


In [ ]:
# Comparison table: baseline row plus all four models, same test set, threshold 0.5
baseline <- data.frame(Model = "Majority baseline (always no)", Accuracy = majority_rate, AUC = 0.5,
                       Recall_yes = 0, Precision_yes = NA_real_)
model_comparison <- rbind(baseline, do.call(rbind, results))
rownames(model_comparison) <- NULL
model_comparison[, -1] <- round(model_comparison[, -1], 3)

cat("Model comparison on the same test set (threshold 0.5; precision undefined for the baseline):\n")
print(model_comparison, row.names = FALSE)

# Plot the AUC of the four models
plot_df <- model_comparison[model_comparison$Model != "Majority baseline (always no)", ]
ggplot(plot_df, aes(x = reorder(Model, AUC), y = AUC)) +
  geom_col(fill = "steelblue", alpha = 0.8) +
  geom_text(aes(label = sprintf("%.3f", AUC)), hjust = -0.1) +
  geom_hline(yintercept = 0.5, linetype = "dashed", colour = "grey40") +
  coord_flip(ylim = c(0.4, 1)) +
  labs(title = "Model comparison by test AUC", x = "Model", y = "Test AUC") +
  theme_minimal()


### Reading the table as a call-centre manager

Read your own table with these questions:

- **Accuracy**: every model sits within one or two points of the baseline. Accuracy cannot separate the models on data with 11 percent subscribers, so "90 percent accuracy" in a report tells the manager almost nothing.
- **AUC** does separate them: the ensembles (forest and XGBoost) should rank customers better than the single tree, with the Lasso in between or close to the forest. The gap between the single tree and the ensembles is what Sections 2 to 4 led us to expect. Which model ranks best in your run?
- **Recall on subscribers** at the 0.5 threshold is low for every model. At this threshold the call list holds only the customers the model is very sure about, so most customers who would say yes are never called. Which model calls the most of them?
- **Precision on subscribers**: of the customers on a model's short list, how many subscribe? Compare that with 11 in 100 for a random call. A model with a longer list (higher recall) usually pays for it with lower precision.

The threshold of 0.5 is a convention, not a business decision. A call centre with the budget to call 20 percent of the customers should lower the threshold until the list has the right length and then compare the models' precision at that list length. Exercise 1 does this.

### Model characteristics, as implemented here

1. **Lasso** (`cv.gamlr`, logistic, standardised internally)
   - Selects features by setting coefficients to zero; the path plot in Section 1 shows the count growing as the penalty weakens
   - Coefficients are readable as directions and sizes on the original scale
   - Fast, including the cross-validation over the whole path
   - Linear in the features: thresholds and interactions must be built by hand
   - With about 26 dummy-encoded features and 33,000 rows there is little to select, so `lambda.min` keeps most of them; `lambda.1se` keeps fewer at almost the same error

2. **Decision Tree** (`rpart`, depth 3, at least 100 customers per leaf, `cp = 0`)
   - Readable as a flowchart; no scaling needed; takes factors directly
   - Finds thresholds and interactions on its own
   - Stops when a split no longer lowers the misclassification count, so it can be smaller than the same-limits scikit-learn tree
   - Overfits badly without limits (Section 2: training AUC near 1 versus a much lower test AUC)
   - Unstable: small changes in the data can change the top split

3. **Random Forest** (`randomForest`, 500 trees, `sqrt(p)` features per split)
   - Averages many deep trees; the variance falls as trees are added (Section 3)
   - Works with the defaults; the out-of-bag error is an honest estimate at no cost
   - Gives an importance ranking, which is not an effect size
   - Not readable; slower; refuses missing values, so they must be filled beforehand

4. **XGBoost** (`xgb.cv` with early stopping, then `xgb.train`)
   - Usually the best AUC; trees are fitted one after another to the remaining errors
   - Early stopping picks the number of rounds from the data
   - Handles missing values natively; fast on large data
   - More parameters to tune (`eta`, `max_depth`, rounds); needs a numeric design matrix

### Choosing a model

- Need to explain the rule to a colleague or a regulator? Tree or Lasso.
- Want the best ranking with little tuning? Random forest.
- Want the best ranking and can tune? XGBoost with early stopping.
- Whatever you pick, report the AUC and the precision and recall at the list length you will actually use, next to the baseline.


## Exercises

1. **A call budget.** The call centre can afford to call 1,000 of the test customers. Rank the test customers by the random forest's predicted probability and select the top 1,000. Deliverable: report how many of them subscribed, how many subscribers a random selection of 1,000 would find on average (subscriber share times 1,000), and one sentence on what this lift means for the manager.
2. **Tree complexity.** Fit trees with `cp` in 0.01, 0.005, 0.001, 0.0005 and 0, with `maxdepth = 30`, `minbucket = 1` and `xval = 0`. Deliverable: a table of the number of leaves, training AUC and test AUC by `cp`, and one sentence on the `cp` at which overfitting starts.
3. **Learning rate.** Rerun `xgb.cv` with `eta` 0.01, 0.1 and 0.3, early stopping on. Deliverable: the best number of rounds and the test AUC for each, plus one sentence on how the learning rate and the number of rounds trade off.
4. **Explain accuracy to a manager.** The shallow tree's accuracy is close to the majority baseline. Deliverable: three sentences, for a manager with no statistics background, on why accuracy is the wrong yardstick on this data and which number in the comparison table to look at instead.
5. **Argue before you code.** The bank asks whether calling a customer more than once in a campaign (`campaign > 1`) raises the chance of a subscription. Before writing any code, argue whether repeated calling is as good as random given the pre-treatment covariates. Name one confounder, and one way in which the outcome itself could drive the treatment. Deliverable: one paragraph and a verdict on whether an analysis like the one in Section 6 could answer the question with this data.


---

# 6. From Prediction to Causal Questions (optional, about 5 minutes runtime)

This section installs one extra package, `grf`. Skip the section if you are short of time; nothing later depends on it.

## Prediction versus intervention

Every model above answers a prediction question: given what the bank knows before the call, how likely is a subscription? The random forest tells us which features matter most for that prediction. It does not tell us what would happen if the bank changed anything.

Take the optional demonstration in Section 3: with `duration` included, the forest says call length is by far the most important feature. Can the bank raise subscriptions by making calls longer? No. Long calls happen because the customer is interested, not the other way round. "Important for prediction" is a statement about correlations in the data as it was generated. "Causes subscription" is a statement about what happens after an intervention, and it needs a different kind of argument.

## A treatment the bank actually sets

The bank chooses the channel for each call: mobile phone (`contact == "cellular"`) or landline (`contact == "telephone"`). That is an intervention, so "does calling on a mobile phone raise subscriptions?" is a causal question the bank could act on.

**What we would need to believe.** To read a comparison of the two channels as a causal effect, two assumptions must hold:

1. *Unconfoundedness given pre-treatment covariates.* Among customers with the same pre-treatment characteristics, the channel was chosen as if at random. Nothing that affects both the channel choice and the subscription decision is left out.
2. *Overlap.* For every kind of customer in the comparison, both channels were actually used. Where one group never received a landline call, there is nothing to compare.

Both are assumptions, not facts we can verify from the data. The channel may itself depend on customer characteristics (which customers gave the bank a mobile number, which period the call took place in), so treat the estimates below as an illustration of the method, not as a result to act on.

**Pre-treatment covariates only.** We adjust for `age`, `marital`, `education`, `housing`, `loan` and the five macro indicators, all fixed before the channel was chosen. We deliberately exclude:

- `duration`, `campaign`: realised during the current campaign, after the channel decision. Conditioning on them would adjust away part of the effect.
- `pdays`, `previous`, `poutcome`: results of earlier campaigns, which were themselves run over a channel. They are partly consequences of past channel choices.

## Plan

1. Naive difference in subscription rates between the two channels.
2. One adjusted estimator: a causal forest (`grf`), whose `average_treatment_effect` is a doubly robust (AIPW) estimate built from out-of-bag propensity and outcome predictions.
3. An overlap check on the estimated propensity scores.


In [ ]:
# One package for this section, installed as a binary from the Posit mirror set in the first cell
# (a source build, if no binary is available, can take 15 to 30 minutes).
if (!requireNamespace("grf", quietly = TRUE)) {
  cat("Installing grf...\n")
  install.packages("grf")
}
suppressPackageStartupMessages(library(grf))
cat("grf ready.\n")


In [ ]:
# Treatment: was the customer called on a mobile phone (1) or a landline (0)?
causal_data <- data.table::copy(data_full)
W <- as.numeric(causal_data$contact == "cellular")
Y <- as.numeric(causal_data$y == "yes")

# Pre-treatment covariates only: fixed before the channel was chosen
pre_treatment <- c("age", "marital", "education", "housing", "loan",
                   "emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed")
X <- model.matrix(~ ., data = causal_data[, ..pre_treatment])[, -1]
cat("Rows:", nrow(X), ", covariates after dummy encoding:", ncol(X), "\n")
cat("Treated (cellular):", sum(W == 1), ", control (telephone):", sum(W == 0), "\n")

# Naive comparison: difference in subscription rates
p1 <- mean(Y[W == 1])
p0 <- mean(Y[W == 0])
naive_ate <- p1 - p0
se_naive <- sqrt(p1 * (1 - p1) / sum(W == 1) + p0 * (1 - p0) / sum(W == 0))
cat(sprintf("\nSubscription rate, cellular:  %.3f\n", p1))
cat(sprintf("Subscription rate, telephone: %.3f\n", p0))
cat(sprintf("Naive difference in means:    %.3f  (SE %.3f)\n", naive_ate, se_naive))
cat("This compares different customers in different periods. It is not yet an effect of the channel.\n")


## Adjusted estimate: causal forest

`causal_forest` first fits two auxiliary forests, one for the propensity score `e(X) = P(cellular | X)` and one for the outcome `E[Y | X]`, using out-of-bag predictions so that no customer's own row is used to predict that customer (the forest analogue of cross-fitting). It then grows a forest whose leaves are chosen to make the treatment effect as different as possible between leaves. `average_treatment_effect` combines the out-of-bag predictions into a doubly robust (AIPW) estimate of the average effect with a standard error: it is consistent if either the outcome model or the propensity model is right, and cross-fitting is what makes the standard error valid when the models are flexible.


In [ ]:
# Fit the causal forest (500 trees keeps the runtime at a few minutes)
set.seed(123)
cf <- causal_forest(X, Y, W, num.trees = 500, seed = 123)

# Out-of-bag propensity estimates from the auxiliary forest
e_hat <- cf$W.hat
cat(sprintf("Propensity model AUC (out of bag): %.3f\n", auc_of(W, e_hat)))
cat("Causal forest fitted.\n")


## Overlap check

The histogram shows the estimated propensity scores for the two groups. Good overlap means that, at every value of the propensity score, both channels appear. If one group piles up at 0 or 1, those customers have no counterpart in the other group, and no estimator can compare them. The AIPW weights `1 / e(X)` and `1 / (1 - e(X))` then explode, so we keep only customers whose propensity lies between 0.05 and 0.95 (the common-support sample) and say clearly that the estimate applies to them only.


In [ ]:
breaks <- seq(0, 1, by = 0.025)
hist(e_hat[W == 0], breaks = breaks, col = rgb(1, 0, 0, 0.5), border = "white",
     main = "Overlap check", xlab = "Estimated propensity score P(cellular | X), out of bag", xlim = c(0, 1))
hist(e_hat[W == 1], breaks = breaks, col = rgb(0, 0, 1, 0.5), border = "white", add = TRUE)
legend("top", c("Telephone (control)", "Cellular (treated)"), fill = c(rgb(1, 0, 0, 0.5), rgb(0, 0, 1, 0.5)))

cat(sprintf("Mean propensity, treated: %.3f; control: %.3f\n", mean(e_hat[W == 1]), mean(e_hat[W == 0])))
common <- e_hat > 0.05 & e_hat < 0.95
cat(sprintf("Customers with propensity <= 0.05: %d; >= 0.95: %d\n", sum(e_hat <= 0.05), sum(e_hat >= 0.95)))
cat(sprintf("Kept in the common-support sample: %d of %d (%.1f%%)\n", sum(common), length(W), 100 * mean(common)))

# Where does the lack of overlap come from? Channel use by period (nr_employed identifies the quarter)
cat("\nCalls by channel and period (rows: nr_employed, a quarterly figure):\n")
print(table(nr_employed = causal_data$nr_employed, cellular = W))


In [ ]:
# Average treatment effect on the common-support sample (doubly robust, out-of-bag nuisance predictions)
ate_cs <- average_treatment_effect(cf, target.sample = "all", subset = common)
naive_cs <- mean(Y[common & W == 1]) - mean(Y[common & W == 0])

cat("Effect of a mobile-phone call versus a landline call on subscription, common-support sample:\n")
cat(sprintf("  Naive difference in means:      %+.3f\n", naive_cs))
cat(sprintf("  Causal forest AIPW estimate:    %+.3f  (SE %.3f, 95%% CI [%+.3f, %+.3f])\n",
            ate_cs["estimate"], ate_cs["std.err"],
            ate_cs["estimate"] - 1.96 * ate_cs["std.err"], ate_cs["estimate"] + 1.96 * ate_cs["std.err"]))
cat(sprintf("\nFor comparison, the naive difference on all customers was %+.3f.\n", naive_ate))


## What did we learn?

Read your own output with these facts from the data in mind:

- The naive comparison on all customers gives about +9.5 percentage points for mobile-phone calls (14.7 versus 5.2 percent subscribing).
- The propensity model predicts the channel very well, and the table above shows why: in one period (`nr_employed` = 5191.0, about 7,800 calls) the bank used landlines only, and in others almost only mobile phones. Overlap fails there, and a large share of the customers falls outside the common-support sample.
- On the common-support sample, compare the naive difference with the causal forest estimate. If adjustment moved the number, the two channels were not used on the same kind of customers even within the overlapping periods.

Questions to ask before believing the adjusted estimate:

- Is a channel effect plausible at all? Perhaps: a mobile call reaches the customer in person, a landline call may reach the household. But which customers gave the bank a mobile number? Customers with a university degree are in the cellular group about seven times in ten, customers with basic education a little over five times in ten. We adjust for age and education, not for income, occupation or how comfortable the customer is with the bank's digital services. Anything of that kind that also affects subscription is a confounder we cannot remove, and the unconfoundedness assumption fails.
- The estimate is for the customers in periods when both channels were used. What is the effect for the others? The data cannot say.
- The macro indicators identify the period, and the period also carries the interest-rate environment and the campaign's history. Adjusting for the period is necessary, but it means we compare channels within a quarter, and a quarter with only a few hundred landline calls carries a lot of weight in the estimate. Does the confidence interval reflect that? Only if the models are right.

What would identify this properly: a randomised assignment of channel (an A/B test). If the bank assigned mobile or landline at random among customers who have both numbers, the naive difference in means would be the causal effect, with no modelling assumptions, and the machinery in this section would only be needed to sharpen the estimate.


### Recommended Reading

- **Book**: "Causal Inference: The Mixtape" by Scott Cunningham (free online), chapters on potential outcomes and matching.
- **Paper**: Athey and Imbens (2019), "Machine Learning Methods That Economists Should Know About", *Annual Review of Economics*.
